## Create the complaint data model 

In [3]:
from pydantic import BaseModel, Field

class Complaint(BaseModel):
    order_id: str = Field(min_length=3)
    customer_name: str = Field(min_length=2)
    product: str = Field(min_length=2)
    amount: float = Field(gt=0)
    complaint: str = Field(min_length=10)

## Create Rahul's complaint

In [4]:
rahul = Complaint(
    order_id="ORD-78452",
    customer_name="Rahul",
    product="Laptop",
    amount=65000,
    complaint="My laptop arrived with a damaged screen."
)

print(rahul.model_dump())

{'order_id': 'ORD-78452', 'customer_name': 'Rahul', 'product': 'Laptop', 'amount': 65000.0, 'complaint': 'My laptop arrived with a damaged screen.'}


## Create a reusable intake function

In [5]:
def submit_complaint(data: dict):
    complaint = Complaint.model_validate(data)

    return {
        "status": "RECEIVED",
        "message": "Complaint submitted successfully",
        "data": complaint.model_dump()
    }


result = submit_complaint(rahul.model_dump())
print(result)

{'status': 'RECEIVED', 'message': 'Complaint submitted successfully', 'data': {'order_id': 'ORD-78452', 'customer_name': 'Rahul', 'product': 'Laptop', 'amount': 65000.0, 'complaint': 'My laptop arrived with a damaged screen.'}}


## Install Dependencies

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## Configure the API key

In [6]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter OpenAI API Key: "
    )

## Define the expected AI output

In [11]:
from pydantic import BaseModel, Field

class ComplaintAnalysis(BaseModel):
    category: str = Field(
        description="Type of customer complaint"
    )
    sentiment: str = Field(
        description="Positive, Negative, or Neutral"
    )
    requested_action: str = Field(
        description="What the customer wants"
    )
    next_step: str = Field(
        description="Suggested next action"
    )

## Create the LangChain model and prompt

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are a customer support complaint analyst.

Analyze the following complaint:

{complaint}

Identify:
- Category
- Sentiment
- Requested action
- Next step

Do not promise refunds or invent order details.
""")

## Create a structured output chain

In [9]:
structured_llm = llm.with_structured_output(
    ComplaintAnalysis
)

analysis_chain = prompt | structured_llm

result = analysis_chain.invoke({
    "complaint": (
        "I purchased a laptop for Rs 65000. "
        "It arrived damaged and I want a refund."
    )
})

print(result.model_dump())

{'category': 'Product Damage', 'sentiment': 'Negative', 'requested_action': 'Refund for damaged laptop', 'next_step': 'Verify the order and product condition, then guide the customer through the refund process according to company policy.'}


## Streamlit interface
- The complaint form and Analyze Complaint with AI button live in `app.py`.
- Run the UI from a terminal using `.venv/bin/python -m streamlit run app.py`.
- This notebook demonstrates the model and chain independently. Run the cells above in order before the next example.

In [10]:
result = analysis_chain.invoke({
    "complaint": "My laptop arrived damaged and I want a refund."
})

print(result.model_dump())

{'category': 'Product Damage', 'sentiment': 'Negative', 'requested_action': 'Refund', 'next_step': 'Verify the damage claim and guide the customer through the return process.'}


## Component 3: Order and document verification with LangGraph
We now use deterministic rules, rather than an LLM, to match a demo order and request missing documents. Attachment presence does not prove authenticity.
Run this notebook from the project folder. The Streamlit UI reuses `verification.py`.


In [ ]:
from verification import verification_graph

complaint_data = rahul.model_dump()
waiting = verification_graph.invoke({"complaint": complaint_data, "invoice_present": False, "photo_present": False})
print(waiting["status"], waiting["message"])


### Resume after receiving documents
For this notebook demonstration, the booleans simulate receiving the files. In Streamlit they come from the upload widgets.


In [ ]:
ready = verification_graph.invoke({"complaint": complaint_data, "invoice_present": True, "photo_present": True})
print(ready["status"], ready["message"])


### What comes next?
Refund eligibility rules, amount-based routing, manager review, and SQLite persistence. No refund is issued by this component.


## Component 4: Simple refund eligibility
**Story:** Rahul has uploaded his invoice and photo. A support agent checks them. Now we apply clear rules.

**Classroom policy (sample only):**
1. Order and attachments must pass Component 3.
2. A support agent must review the documents and confirm damage.
3. Damage must be reported within 7 days of delivery.
4. Amount ≤ ₹10,000 → automatic refund path. Amount > ₹10,000 → manager approval.

The function only returns a decision. It does not issue a refund. The checkboxes in the UI simulate a support agent for teaching; they are not authenticated approvals.


In [ ]:
from eligibility import check_refund_eligibility

# Rahul: ₹65,000, reported after 2 days, documents reviewed.
rahul_decision = check_refund_eligibility(
    amount=65000,
    days_since_delivery=2,
    damage_confirmed=True,
    documents_reviewed=True,
    verification_status=ready["status"],
)
print(rahul_decision)


### Try a smaller amount
Change only the amount. Predict the result before running the cell.


In [ ]:
small_order_decision = check_refund_eligibility(
    amount=5000,
    days_since_delivery=2,
    damage_confirmed=True,
    documents_reviewed=True,
    verification_status="READY_FOR_ELIGIBILITY",
)
print(small_order_decision)


### Exercise
Try ₹10,000, ₹10,001, 7 days, 8 days, and `documents_reviewed=False`. Explain which `if` returns the answer.

**Next component:** connect this function to a LangGraph node, then add manager review.


## Component 5: Manager review and final resolution
**Story:** Rahul passes the demo eligibility rules. His ₹65,000 request needs a manager.

1. The manager reads the complaint and analysis.
2. They choose Approve or Reject and enter their name.
3. Rejection requires a reason.
4. The function closes the demo case and creates a customer message preview.

This is classroom role-play. No authenticated manager, payment, or email service is connected.


In [ ]:
from resolution import resolve_complaint

# Use the Rahul complaint and eligibility result from earlier cells.
approved = resolve_complaint(
    complaint=rahul.model_dump(),
    eligibility=rahul_decision,
    decision="Approve",
    manager_name="Demo Manager",
    reason="Invoice and damage photo reviewed in the demo.",
)
print(approved["status"])
print(approved["customer_message"])


### Try rejecting the request
This is a separate example, not a second decision applied to the closed case.


In [ ]:
rejected = resolve_complaint(
    complaint=rahul.model_dump(),
    eligibility=rahul_decision,
    decision="Reject",
    manager_name="Demo Manager",
    reason="The submitted photo does not establish the reported damage.",
)
print(rejected["customer_message"])


### Automatic path for a smaller order
Use Riya’s second demo order to show the other branch in Streamlit.


In [ ]:
riya = {"order_id": "ORD-10001", "customer_name": "Riya", "product": "Headphones", "amount": 5000, "complaint": "My headphones arrived damaged."}
riya_check = verification_graph.invoke({"complaint": riya, "invoice_present": True, "photo_present": True})
riya_eligibility = check_refund_eligibility(5000, 2, True, True, riya_check["status"])
riya_resolution = resolve_complaint(riya, riya_eligibility)
print(riya_resolution["customer_message"])


### Teach the complete demo
Complaint → AI analysis → Order/document check → Eligibility → Automatic path OR Manager review → Final resolution.

In Streamlit, upload sample files, confirm the two demo review checkboxes, and check eligibility. Try Approve, Reject, and missing rejection notes. Changing documents or eligibility inputs clears the previous decision. New complaint submissions start a new case.

The app keeps results in session state and offers a downloadable JSON case summary. SQLite persistence, durable LangGraph interrupts, real document verification, manager authentication, payments and customer notifications remain future integrations.
